# The balanced Richardson number $R_{ib}$

This notebook demonstrates the new `R_ib` field added to
`dbof.preprocessing.calculated_fields_at_depth.balanced_richardson_number_3d`.

The **balanced Richardson number** (Thomas, Tandon & Mahadevan 2013) is the
gradient Richardson number of a flow in thermal-wind balance — a
dimensionless measure of frontal stability:

$$R_{ib} = \frac{N^2\,f^2}{|\nabla_h b|^2}$$

where $N^2$ is the buoyancy frequency squared, $f$ the Coriolis parameter,
and $|\nabla_h b|^2=(\partial_x b)^2+(\partial_y b)^2$ the squared horizontal
buoyancy-gradient magnitude.

Both $N^2$ and $|\nabla_h b|^2$ are built from the **same** physically
consistent, unscaled buoyancy $b=(g/\rho_0)\rho$ (not the $\times10^3$-scaled
`grad_b2_3d`), so the ratio is genuinely dimensionless. $R_{ib}$ is `NaN`
where $|\nabla_h b|^2=0$, tends to $0$ at the equator ($f\to0$), and
statically unstable columns ($N^2<0$) are **floored to $N^2=0$** so
$R_{ib}=0$ there rather than negative.

It lives in the **DEPTH** pipeline's `mixing_parameters` subset (alongside
`Fr`, `Ro`, `Bu`) and is reduced to 2-D through the standard depth
strategies (`R_ib_sfc`, `R_ib_z25m`, `R_ib_mld`, `R_ib_mld_mean`).

In [ ]:
%matplotlib inline
from IPython.display import display
import numpy as np
import dask.array as da
import xarray as xr
import matplotlib.pyplot as plt

import dbof.preprocessing.calculated_fields_at_depth as cfad
from dbof.preprocessing.physical_constants import OMEGA_EARTH

## Part 1 — Formula demo on synthetic data (offline)

`balanced_richardson_number_3d` accepts pre-computed $N^2$ (`n2`) and
$|\nabla_h b|^2$ (`gradb2`) through keywords, so we can exercise the
closed-form combination without standing up an xgcm grid (only latitude
`YC` is needed, for $f$). This is the same injection pattern the unit tests
(`tests/test_calculated_fields_at_depth.py`) use.

In [ ]:
def rib_eval(n2_1d, gradb2_1d, lat_1d):
    """R_ib over 1-D inputs via the production field function.

    Each argument is a 1-D numpy array (broadcast to a common length);
    they are packed onto a (face=1, j=1, i=N) synthetic tile and passed
    through ``balanced_richardson_number_3d`` exactly as the pipeline
    would call it.  Returns a 1-D numpy array of R_ib.
    """
    n2_1d, gradb2_1d, lat_1d = np.broadcast_arrays(
        n2_1d, gradb2_1d, lat_1d)
    n = n2_1d.size
    ds = xr.Dataset(
        {"YC": (("face", "j", "i"), lat_1d.reshape(1, 1, n))})
    n2 = xr.DataArray(
        da.from_array(n2_1d.reshape(1, 1, n)), dims=("face", "j", "i"))
    g2 = xr.DataArray(
        da.from_array(gradb2_1d.reshape(1, 1, n)), dims=("face", "j", "i"))
    out = cfad.balanced_richardson_number_3d(
        ds, grid=None, n2=n2, gradb2=g2)
    return out.values.ravel()


# Quick sanity check at 30 N (matches the unit test values).
f30 = 2.0 * OMEGA_EARTH * np.sin(np.deg2rad(30.0))
print(f"Coriolis f at 30 N = {f30:.3e} s^-1")
print("R_ib(N^2=1e-5, |grad b|^2=1e-9, 30N) =",
      rib_eval(np.array([1e-5]), np.array([1e-9]),
               np.array([30.0]))[0])

In [ ]:
# Figure 1: inverse-square dependence on the horizontal
# buoyancy-gradient magnitude, at fixed latitude (30 N).
lat = 30.0
gradb = np.logspace(-8, -6, 200)        # |grad_h b|  [s^-2]
gradb2 = gradb ** 2                       # [s^-4]

fig, ax = plt.subplots(figsize=(7, 5))
for n2v in [1e-6, 1e-5, 1e-4]:            # weak -> strong stratification
    rib = rib_eval(np.full_like(gradb2, n2v), gradb2,
                   np.full_like(gradb2, lat))
    ax.loglog(gradb, rib, label=f"$N^2$={n2v:g} s$^{{-2}}$")
ax.axhline(1.0, color="0.7", lw=0.8, ls="--")   # R_ib = 1 reference
ax.set_xlabel(r"$|\nabla_h b|$  [s$^{-2}$]")
ax.set_ylabel(r"$R_{ib}$  (dimensionless)")
ax.set_title(r"$R_{ib}\propto |\nabla_h b|^{-2}$  (at 30 N)")
ax.legend()
fig.tight_layout()
display(fig)
plt.close(fig)

$R_{ib}$ also scales as $f^2$, so it vanishes at the
equator. The next figure shows that latitude dependence, and verifies the
**$N^2<0$ floor**: a statically unstable column is clamped to $N^2=0$, giving
$R_{ib}=0$ at every latitude.

In [ ]:
# Figure 2: f^2 dependence (-> 0 at the equator) and the
# negative-N^2 floor (unstable column -> R_ib = 0 everywhere).
lats = np.linspace(-60.0, 60.0, 241)
n2v, gradb2v = 1.0e-5, 1.0e-13

rib_stable = rib_eval(np.full_like(lats, n2v),
                      np.full_like(lats, gradb2v), lats)
rib_unstable = rib_eval(np.full_like(lats, -n2v),     # N^2 < 0 -> floored
                        np.full_like(lats, gradb2v), lats)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(lats, rib_stable, label="stable ($N^2>0$)")
ax.plot(lats, rib_unstable, label="unstable ($N^2<0$, floored to 0)")
ax.axvline(0, color="0.7", lw=0.8)
ax.set_xlabel("latitude  [deg N]")
ax.set_ylabel(r"$R_{ib}$  (dimensionless)")
ax.set_title(r"$R_{ib}\propto f^2$ ; unstable columns floored to 0")
ax.legend()
fig.tight_layout()
display(fig)
plt.close(fig)

## Part 2 — A real LLC4320 tile (requires network / S3)

This mirrors
`tests/test_calculated_fields_at_depth.test_balanced_richardson_against_real_tile`:
load one 720×720 tile, build a single-face xgcm grid, and compute the
**surface** and **25 m** balanced Richardson number via the production
function + `apply_depth_strategies`. $R_{ib}$ needs only `Theta`/`Salt`
(no velocities).

It is wrapped in `try/except` so the notebook still runs end-to-end offline
(the cell simply reports that it was skipped).

In [ ]:
def load_tile_rib():
    """Compute surface and 25 m R_ib on one real LLC4320 tile.

    Returns (XC, YC, R_ib_sfc, R_ib_z25m) as 2-D numpy arrays.
    Requires network access to the LLC4320 S3 store.
    """
    import dbof.tiles.tile_utils as tu
    from dbof.tiles.tile_mapping import rect_ij_to_tile
    from dbof.llc4320_ingestion.grid import set_xgcm_grid
    from dbof.preprocessing.depth_strategies import \
        apply_depth_strategies

    s3 = tu._resolve_s3_source(None)
    timestamp = "2012-11-09 12:00:00"
    tile = rect_ij_to_tile(5000, 5000)        # open-ocean tile (face 4)

    ds_grid = tu._load_grid_for_tile(s3, tile)
    ds_tracers = tu._load_tracers_for_tile(
        s3, timestamp, tile, vars_needed=["Theta", "Salt"])

    # The tracer loader slices only i/j; slice the staggered dims to the
    # same tile extent so a clean single-face xgcm grid can be built.
    ds_grid = ds_grid.isel(
        i_g=tile.i_face_slice, j_g=tile.j_face_slice)
    ds_merge = xr.merge([ds_grid, ds_tracers])

    vvars = [v for v in ("Z", "Zl", "Zu", "Zp1", "drF") if v in ds_grid]
    grid = set_xgcm_grid(ds_grid.drop_vars(vvars),
                         use_connections=False)

    import dask
    rib_3d = cfad.balanced_richardson_number_3d(ds_merge, grid)
    # Surface and 25 m depth strategies (neither needs the MLD).
    res = apply_depth_strategies(
        rib_3d, "R_ib", ds_merge, mld=None,
        requested={"R_ib_sfc", "R_ib_z25m"})
    # A single dask.compute() so the shared full-column N²/gradient graph
    # is evaluated once for both depth slices (mirrors the pipeline's
    # _materialise_results) — computing each separately would double the
    # cost.
    sfc, z25 = dask.compute(
        res["R_ib_sfc"], res["R_ib_z25m"], retries=10)
    sfc = sfc.isel(face=0)
    z25 = z25.isel(face=0)
    g2 = ds_grid.isel(face=0)
    return g2["XC"].values, g2["YC"].values, sfc.values, z25.values


try:
    XC, YC, RIB_SFC, RIB_Z25 = load_tile_rib()
    have_tile = True
except Exception as exc:        # offline / no S3 access
    have_tile = False
    print(f"Skipping real-tile demo (no network?): {exc!r}")

In [ ]:
if have_tile:
    # R_ib spans many orders of magnitude -> view log10(R_ib).  R_ib >= 0
    # after the N^2 floor, so mask non-positive cells before the log.
    def _log10_pos(a):
        return np.log10(np.where(a > 0, a, np.nan))

    fig, axes = plt.subplots(1, 2, figsize=(12, 5),
                             sharex=True, sharey=True)
    for ax, fld, ttl in [(axes[0], RIB_SFC, "surface"),
                         (axes[1], RIB_Z25, "25 m")]:
        la = _log10_pos(fld)
        lim_lo = np.nanquantile(la, 0.02)
        lim_hi = np.nanquantile(la, 0.98)
        pcm = ax.pcolormesh(la, cmap="viridis", vmin=lim_lo, vmax=lim_hi)
        ax.set_title(f"$\\log_{{10}} R_{{ib}}$  ({ttl})")
        ax.set_aspect("equal")
        fig.colorbar(pcm, ax=ax, shrink=0.8)
    fig.suptitle("Balanced Richardson number on one LLC4320 tile")
    fig.tight_layout()
    display(fig)
    plt.close(fig)

    finite = np.isfinite(RIB_SFC[5:-5, 5:-5])
    print("surface R_ib: finite fraction =",
          round(float(finite.mean()), 4),
          " median =",
          round(float(np.nanmedian(RIB_SFC[5:-5, 5:-5])), 2))
else:
    print("Real-tile panels skipped (Part 2 needs network access).")

### Takeaways

* `R_ib` is wired into the **`mixing_parameters`** subset and registered in
  `dbof.defs.fields_dmodel` (units: dimensionless).
* Request it through any depth strategy, e.g. `R_ib_sfc`, `R_ib_z25m`,
  `R_ib_mld`, `R_ib_mld_mean`.
* It scales as $N^2 f^2 / |\nabla_h b|^2$: large where stratification is
  strong relative to the front, small at sharp fronts, and floored to $0$
  in statically unstable columns ($N^2<0$).